In [1]:
import cv2
import mediapipe as mp
import pyautogui 
import time
import subprocess

mphands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hand = mphands.Hands(max_num_hands=1)

video = cv2.VideoCapture(0)
tipids = [4,8,12,16,20]

last_action = 0
cooldown = 1.3

while True:
    suc,img = video.read()
    if not suc:
        break

    img = cv2.flip(img,1)
    img1 = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    result = hand.process(img1)
    lmlist=[]
    gesture="No Hand"

    if result.multi_hand_landmarks:
        handlm = result.multi_hand_landmarks[0]

        for id, lm in enumerate(handlm.landmark):
            cx=lm.x
            cy=lm.y
            lmlist.append([id,cx,cy])

        if len(lmlist)==21:
            fingerlist=[]

            # thumb
            if lmlist[8][1] < lmlist[20][1]:
                fingerlist.append(0 if lmlist[4][1] > lmlist[3][1] else 1)
            else:
                fingerlist.append(0 if lmlist[4][1] < lmlist[3][1] else 1)

            # other fingers
            for i in range(1,5):
                if lmlist[tipids[i]][2] > lmlist[tipids[i]-2][2]:
                    fingerlist.append(0)
                else:
                    fingerlist.append(1)

            now = time.time()

            if now - last_action > cooldown:

                # Palm → Chrome
                if fingerlist == [1,1,1,1,1]:
                    gesture = "Open Chrome"
                    subprocess.Popen("start chrome", shell=True)
                    last_action = now

                # Victory → Calculator
                elif fingerlist == [0,1,1,0,0]:
                    gesture = "Open Calculator"
                    subprocess.Popen("calc")
                    last_action = now

                # Thumbs up → Notepad
                elif fingerlist == [1,0,0,0,0]:
                    gesture = "Open Notepad"
                    subprocess.Popen("notepad")
                    last_action = now

                # Index finger only → YouTube
                elif fingerlist == [0,1,0,0,0]:
                    gesture = "Open YouTube"
                    subprocess.Popen("start https://www.youtube.com", shell=True)
                    last_action = now

                # Fist → Close app
                elif fingerlist == [0,0,0,0,0]:
                    gesture = "Close App"
                    pyautogui.hotkey("alt", "f4")
                    last_action = now

                else:
                    gesture = "Unknown"

        mp_drawing.draw_landmarks(img, handlm, mphands.HAND_CONNECTIONS)

    cv2.putText(img, gesture, (30,80), cv2.FONT_HERSHEY_COMPLEX, 1.4, (0,255,0), 2)

    cv2.imshow("Gesture App Launcher", img)

    if cv2.waitKey(1) & 0XFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()